# ⑤ 평가 — 네 가지로 나눠 채점
담당: 평가 · 코드: `rag/evaluation/evaluation.py`

04 생성의 `artifacts/answers.jsonl` 하나를 네 가지로 채점해, 나쁜 답이 검색에서 왔는지 · 생성에서 왔는지 가릅니다.
RAGAS 심판은 아래 설정 셀에서 선택한 모델(OpenAI API, 유료)입니다. 각 표의 마지막 행(`평균`)이 요약입니다.
04 를 Langfuse 키와 함께 실행했다면 질문별 점수가 그 질문의 Langfuse 추적에도 붙습니다 (지연 시간 · 토큰 · 비용은 추적에 이미 있어 점수로 보내지 않음).

| 평가 | 판단하는 것 | 저장 파일 |
|---|---|---|
| Retrieval | 검색한 청크가 정답에 필요한 내용을 담았나 | `artifacts/retrieval.csv` |
| Generation | 답이 질문에 맞고 핵심 사실을 담았나 | `artifacts/generation.csv` |
| Grounding | 답이 검색한 청크 안에서만 나왔나 (환각) | `artifacts/grounding.csv` |
| End-to-End | 최종 답이 정답과 맞나 · 시간 · 토큰 · 비용 | `artifacts/end_to_end.csv` |

In [ ]:
from rag.config import ARTIFACTS, load_config
from rag.evaluation.evaluation import (evaluate_end_to_end, evaluate_generation, evaluate_grounding,
                                       evaluate_retrieval)
from rag.usage import usage_summary

cfg = load_config()
ANSWERS = ARTIFACTS / "answers.jsonl"

## 심판 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
이 설정은 네 가지 RAGAS 평가에 공통으로 적용됩니다. 평가할 때 OpenAI API 비용이 발생합니다.


In [ ]:
JUDGE_MODEL = cfg["evaluation"]["judge_model"]                     # 예: "gpt-5-mini"
JUDGE_REASONING_EFFORT = cfg["evaluation"]["judge_reasoning_effort"]  # 예: "low"
JUDGE_MAX_TOKENS = cfg["evaluation"]["judge_max_tokens"]             # 출력·추론 토큰 한도
JUDGE_PRICES = cfg["evaluation"]["judge_usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가

cfg["evaluation"].update(judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT,
                         judge_max_tokens=JUDGE_MAX_TOKENS, judge_usd_per_1m_tokens=JUDGE_PRICES)
cfg["evaluation"]


## Retrieval
- `context_precision`: 검색한 청크 중 정답에 쓸모 있는 청크가 위에 있나 (RAGAS)
- `context_recall`: 정답의 내용이 검색한 청크에 다 있나 (RAGAS)
- `key_facts_in_context`: 핵심 사실 중 검색한 청크에 들어 있는 비율

In [ ]:
retrieval = evaluate_retrieval(ANSWERS, cfg)
retrieval

## Generation
- `answer_relevancy`: 답이 질문에 맞는 답인가 (RAGAS, KURE-v1 임베딩)
- `key_facts_in_answer`: 핵심 사실 중 답에 들어 있는 비율

In [ ]:
generation = evaluate_generation(ANSWERS, cfg)
generation

## Grounding
- `faithfulness`: 답의 주장 중 검색한 청크로 뒷받침되는 비율 (RAGAS) — 낮으면 환각

In [ ]:
grounding = evaluate_grounding(ANSWERS, cfg)
grounding

## End-to-End
- `factual_correctness`: 답과 정답의 사실 일치 F1 (RAGAS)
- `latency_s`: 질문 하나의 재작성 + 검색 + 생성 시간(초)
- `input_tokens` · `cached_input_tokens` · `output_tokens` · `reasoning_tokens`: 답변 생성 토큰 (캐시 입력은 입력, 추론은 출력에 포함)
- `rewrite_*_tokens`: 질문 재작성 토큰
- `answer_cost_usd` · `rewrite_cost_usd`: 호출 당시 단가로 계산해 04에서 저장한 비용
- `cost_usd`: rewrite + 답변 생성 예상 비용 합계. OCR·평가 비용은 `api_usage.csv`에서 별도로 확인하세요.

In [ ]:
end_to_end = evaluate_end_to_end(ANSWERS, cfg)
end_to_end

## Judge / RAGAS API 사용량
네 평가 함수가 점수 CSV를 자동 저장합니다. RAGAS 내부 Judge 호출은 평가 종류별로 `artifacts/api_usage.csv`에 기록하며, 아래 표는 누적 사용량입니다. RAGAS와 Judge 비용을 중복해서 더하지 않습니다.

In [ ]:
usage_summary("judge")